# 03 · The LSTM as a Tool for an LLM

The trained LSTM becomes a **function the LLM can call**. The LLM decides when to run it, reads the structured
result and explains it — optionally grounded in documents retrieved from a **persistent local Chroma** vector store.

1. Wrap the model as a tool and run a minimal tool-calling loop
2. Add RAG: ingest `docs/ecg_reference.md` into Chroma and expose search as a second tool
3. Use the same `Agent` the FastAPI app serves
4. Call the running FastAPI service over HTTP: the Tools API (no LLM) and the Agent API (with its step trace)

**OpenAI (`gpt-4o-mini`) is the agentic core**: it decides which tool to call and writes the answer. Our LSTM is one
of its tools. If the OpenAI call fails, the agent switches to Gemini (`gemini-2.5-flash`) through Google's
OpenAI-compatible endpoint. Requires `OPENAI_API_KEY` (and optionally `GEMINI_API_KEY`) in `.env`
(see `.env.example`) and the model from `02_lstm_training.ipynb`. All recordings used here are DS2 test patients
that the model never saw during training.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data" / "mitdb"
MODEL_DIR = ROOT / "models"

In [2]:
import json
import numpy as np

from openai import OpenAI

from app.config import get_settings
from app.ecg import mitbih
from app.ecg.classifier import ECGClassifier

settings = get_settings()
endpoint = settings.llm_endpoint()          # LLM_PROVIDER=openai -> OPENAI_API_KEY, gpt-4o-mini
llm = OpenAI(api_key=endpoint.api_key, base_url=endpoint.base_url)
clf = ECGClassifier(settings.resolve(settings.ecg_model_path), settings.resolve(settings.ecg_meta_path))
print(f"LLM: {endpoint.provider}:{endpoint.model} | ECG classes: {clf.classes}")

I0000 00:00:1794200575.739805 2441583 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


LLM: openai:gpt-4o-mini | ECG classes: ['N', 'S', 'V']


## 1. A minimal tool: `analyze_ecg`
The LLM cannot pass 20,000 samples as an argument, so recordings live in a registry and the tool takes an id.

In [3]:
mitbih.download(DATA_DIR)
recordings = {}
for rec_id, seconds in [("233", 60), ("100", 60)]:
    sig, _, _, fs = mitbih.load_record(DATA_DIR, rec_id)
    recordings[f"mitdb_{rec_id}"] = {"signal": sig[: seconds * fs], "fs": fs}

def analyze_ecg(ecg_id: str) -> dict:
    r = recordings.get(ecg_id)
    if r is None:
        return {"error": f"unknown ecg_id {ecg_id}", "available": list(recordings)}
    return clf.analyze(r["signal"], r["fs"])

# Direct call — this is exactly what the LLM will receive
print(json.dumps({k: v for k, v in analyze_ecg("mitdb_233").items() if k != "abnormal_beats"}, indent=2))

{
  "duration_s": 60.0,
  "n_beats_classified": 102,
  "heart_rate_bpm": 103.4,
  "rr_sdnn_ms": 116.0,
  "beat_counts": {
    "N": 73,
    "S": 0,
    "V": 29
  },
  "beat_percentages": {
    "N": 71.57,
    "S": 0.0,
    "V": 28.43
  },
  "class_descriptions": {
    "N": "Normal / bundle branch block / escape beat",
    "S": "Supraventricular ectopic beat",
    "V": "Ventricular ectopic beat"
  },
  "abnormal_beats_truncated": false,
  "signal_quality": {
    "ok": true,
    "low_quality_fraction": 0.0,
    "median_kurtosis": 4.63,
    "warning": null
  },
  "disclaimer": "Automated research model output; not a medical diagnosis."
}


In [4]:
ECG_TOOL = {
    "type": "function",
    "function": {
        "name": "analyze_ecg",
        "description": "Run the LSTM ECG beat classifier on a stored recording. Returns heart rate, RR variability, AAMI beat counts and abnormal beat times.",
        "parameters": {
            "type": "object",
            "properties": {"ecg_id": {"type": "string", "enum": list(recordings)}},
            "required": ["ecg_id"],
        },
    },
}

def run_with_tools(question: str, tools: list, handlers: dict, system: str, max_steps: int = 5) -> str:
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        msg = llm.chat.completions.create(model=endpoint.model, messages=messages, tools=tools).choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            print(f"→ LLM called {tc.function.name}({args})")
            result = handlers[tc.function.name](**args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return "max steps reached"

answer = run_with_tools(
    "Compare recordings mitdb_233 and mitdb_100. Which one shows more ectopic activity and what kind?",
    tools=[ECG_TOOL],
    handlers={"analyze_ecg": analyze_ecg},
    system="You are a cardiology research assistant. Use the analyze_ecg tool; never guess. Results are not a diagnosis.",
)
print(answer)

→ LLM called analyze_ecg({'ecg_id': 'mitdb_233'})
→ LLM called analyze_ecg({'ecg_id': 'mitdb_100'})


The analysis of the ECG recordings shows the following results:

### Recording: mitdb_233
- **Duration:** 60 seconds
- **Heart Rate:** 103.4 bpm
- **RR Variability (SDNN):** 116 ms
- **Total Beats Classified:** 102
- **Normal Beats (N):** 73 (71.57%)
- **Ventricular Ectopic Beats (V):** 29 (28.43%)
- **Abnormal Beats:** 
  - There are 29 ventricular ectopic beats, occurring at various times during the recording with high confidence.

### Recording: mitdb_100
- **Duration:** 60 seconds
- **Heart Rate:** 73.9 bpm
- **RR Variability (SDNN):** 37.6 ms
- **Total Beats Classified:** 73
- **Normal Beats (N):** 73 (100%)
- **Ventricular Ectopic Beats (V):** 0 (0%)
- **Abnormal Beats:** 
  - No abnormal beats detected.

### Comparison Summary
- **Mitdb_233:** Shows significant ectopic activity with **29 ventricular ectopic beats**.
- **Mitdb_100:** Displays no ectopic activity with **0 abnormal beats**.

### Conclusion
**Mitdb_233** demonstrates more ectopic activity compared to **mitdb_100**, 

## 2. Add RAG over the persistent Chroma store
Embeddings default to a local MiniLM model (no API key). Data persists in `chroma_db/`.

In [5]:
from app.embeddings import Embedder
from app.ingest import chunk_text
from app.vector_store import VectorStore

store = VectorStore(settings, Embedder(settings))
text = (ROOT / "docs" / "ecg_reference.md").read_text()
chunks = chunk_text(text, settings.chunk_size, settings.chunk_overlap)
store.delete_source("ecg_reference.md")
print("Ingested chunks:", store.add_chunks("ecg_reference.md", chunks), "| collection size:", store.count())

for hit in store.search("What does a wide QRS complex indicate?", top_k=3):
    print(f"{hit['score']:.3f}  {hit['text'][:120]!r}")

Ingested chunks: 5 | collection size: 5


0.502  '- **N (normal and bundle branch block beats):** normal sinus beats, left bundle branch block (LBBB),\n  right bundle bran'
0.460  '## Normal ECG intervals (adult)\n\n- Heart rate: 60–100 bpm at rest. Below 60 bpm is bradycardia; above 100 bpm is tachyca'
0.362  'followed by a compensatory pause.\n- **F (fusion beats):** a fusion of a ventricular and a normal beat; morphology is int'


In [6]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Semantic search over cardiology reference documents.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
    },
}

answer = run_with_tools(
    "Analyze mitdb_233. Is the amount of ventricular ectopy clinically significant according to the reference documents?",
    tools=[ECG_TOOL, SEARCH_TOOL],
    handlers={"analyze_ecg": analyze_ecg, "search_knowledge_base": lambda query: {"results": store.search(query, top_k=4)}},
    system="You are a cardiology research assistant. Use analyze_ecg for recordings and search_knowledge_base for clinical thresholds; cite sources. Not a diagnosis.",
)
print(answer)

→ LLM called analyze_ecg({'ecg_id': 'mitdb_233'})
→ LLM called search_knowledge_base({'query': 'ventricular ectopy clinical significance thresholds'})


The analysis of the ECG recording from `mitdb_233` showed the following results:

- **Total Duration**: 60 seconds
- **Heart Rate**: 103.4 bpm
- **RR Variability (SDNN)**: 116 ms
- **Total Beats Classified**: 102
- **Beat Counts**:
  - Normal (N): 73 beats (71.57%)
  - Supraventricular Ectopic (S): 0 beats (0.0%)
  - Ventricular Ectopic (V): 29 beats (28.43%)

### Abnormal Beats
A total of 29 ventricular ectopic beats were identified during the recording. The abnormal beats were characterized by high confidence scores, indicating a reliable classification of premature ventricular contractions (PVCs).

### Clinical Significance of Ventricular Ectopy
According to the clinical reference documents reviewed:
- Occasional PVCs and atrial premature beats are common and can occur in healthy individuals.
- However, frequent ectopy is concerning. A commonly used threshold is more than **10% of beats**, which corresponds to more than about 10,000 PVCs per 24 hours. When ventricular ectopy exceeds

## 3. The production `Agent` (same code the FastAPI `/agent/chat` endpoint runs)


In [7]:
from app.agent import Agent, ECGRegistry

registry = ECGRegistry()
sig, _, _, fs = mitbih.load_record(DATA_DIR, "221")
rec = registry.add(sig[: 60 * fs].tolist(), fs, name="mitdb_221 first minute")

agent = Agent(settings, store, registry, clf)
out = agent.chat("Summarise this ECG and explain any abnormal beats in plain language.", ecg_id=rec.id)
print("LLM:", out["llm"])
print("Tools used:", out["tool_calls"])
print("Sources:", out["sources"])
print(out["answer"])

LLM: {'provider': 'openai', 'model': 'gpt-4o-mini', 'fallback_used': False}
Tools used: [{'tool': 'analyze_ecg', 'arguments': {'ecg_id': '5109d5972d23'}}]
Sources: []
The analysis of the ECG recording indicates the following:

- **Duration**: 60 seconds
- **Heart Rate**: 75.2 beats per minute
- **Total Beats Classified**: 74
- **Normal Beats (N)**: 63 (85.14%)
- **Supraventricular Ectopic Beats (S)**: 2 (2.7%)
- **Ventricular Ectopic Beats (V)**: 9 (12.16%)

### Abnormal Beats Explained:
1. **Ventricular Ectopic Beats (V)**: 
   - These are beats that originate from the ventricles instead of the normal pacemaker in the heart. They can occur occasionally and are often benign, but frequent occurrences may require further evaluation. In this recording, there are several ventricular ectopic beats detected at the following times:
     - 1.675 seconds (confidence: 99.8%)
     - 6.997 seconds (confidence: 99.8%)
     - 18.086 seconds (confidence: 99.9%)
     - 21.356 seconds (confidence: 99.9

## 4. Through the FastAPI service: Tools API and Agent API

Start the server in a terminal from the project root:

```bash
source env/bin/activate
uvicorn app.main:app --reload
```

In [8]:
import httpx

API = f"http://localhost:{settings.app_port}"
try:
    with httpx.Client(base_url=API, timeout=120) as http:
        print(http.get("/health").json())
        sig, _, _, fs = mitbih.load_record(DATA_DIR, "214")
        ecg_id = http.post("/ecg/upload", json={"signal": sig[: 30 * fs].tolist(), "fs": fs, "name": "mitdb_214"}).json()["ecg_id"]

        # Tools API: the same tool the agent uses, called directly without an LLM
        print("Tools:", [t["name"] for t in http.get("/tools").json()])
        tool = http.post("/tools/analyze_ecg/invoke", json={"arguments": {"ecg_id": ecg_id}}).json()
        print("Direct tool call:", tool["result"]["beat_counts"], f"({tool['duration_ms']:.0f} ms)")

        # Agent API: OpenAI decides to call the tool, then explains the result
        r = http.post("/agent/chat", json={"message": "What does this recording show? Is it concerning?", "ecg_id": ecg_id}).json()
        for s in r["steps"]:
            print(f"  {s['type']:4} {s.get('tool') or s['decision']:<14} {s['duration_ms']:>7.0f} ms")
        print(r["llm"]); print(r["answer"])
except httpx.ConnectError:
    print(f"API not running at {API} — start it with `uvicorn app.main:app` and re-run this cell.")


{'status': 'ok', 'vector_collection': 'ecg_knowledge', 'ecg_model_loaded': True, 'llm': {'provider': 'openai', 'model': 'gpt-4o-mini', 'fallback': 'gemini:gemini-2.5-flash'}}
Tools: ['search_knowledge_base', 'analyze_ecg', 'list_ecg_recordings']


Direct tool call: {'N': 34, 'S': 0, 'V': 2} (1594 ms)


  llm  call_tools        2888 ms
  tool analyze_ecg        119 ms
  llm  answer            5490 ms
{'provider': 'openai', 'model': 'gpt-4o-mini', 'fallback_used': False}
The ECG recording shows the following results:

- **Heart Rate**: Approximately 74.7 beats per minute (bpm).
- **RR Variability**: The standard deviation of the RR intervals is 99.9 milliseconds, indicating some variability in the time between heartbeats.
- **Beat Classification**:
  - **Normal Beats (N)**: 34 beats (94.44% of total)
  - **Ventricular Ectopic Beats (V)**: 2 beats (5.56% of total)
  - **Supraventricular Ectopic Beats (S)**: 0 beats (0% of total)

### Abnormal Beats:
- Two ventricular ectopic beats were detected:
  - The first occurred at approximately 5.97 seconds into the recording with a confidence of 96.3%.
  - The second occurred at approximately 9.06 seconds with a confidence of 99.6%.

### Interpretation:
- The majority of the beats are classified as normal, which is a positive sign.
- The presenc